# exp04 gSDE — 좌/우 편향 점검

다중 seed에서 좌우 도달률 편향이 나타나는지 수치로 확인한다. `eval_results.pkl`(policy_eval.py 산출)만 사용.

- 왼 = goal 각도 +15°~+165°, 오 = −15°~−165° (전 거리·전 지형 seed 평균)
- 기대: 과거 단일 seed(exp00-05)의 극단 편향은 seed 우연 → **다중 seed에선 좌우 차이 미미**

> 실행: 이 노트북을 `analysis/exp04_gsde/` 에서 열어 위→아래로 실행.

In [ ]:
import pickle
import numpy as np

# 이 노트북과 같은 폴더의 eval_results.pkl (policy_eval.py 산출: 순수 배열이라 env import 불필요)
d = pickle.load(open("eval_results.pkl", "rb"))
ang = np.array(d["angles"])
Lidx = [i for i, a in enumerate(ang) if 15 <= a <= 165]     # 왼(+각도)
Ridx = [i for i, a in enumerate(ang) if -165 <= a <= -15]   # 오(-각도)

print("각도:", list(ang))
print("그룹:", {k: len(v) for k, v in d["group_names"].items()})
print("거리:", d["dists"], "| 평가 지형 seed:", d["seeds"])

In [ ]:
def left_right(name):
    """전 거리·전 지형 평균 좌(+)/오(-) 도달률 %."""
    arr = d["results"][name]                     # [평가seed, 거리, 각도] bool
    return 100 * arr[:, :, Lidx].mean(), 100 * arr[:, :, Ridx].mean()

for label, names in d["group_names"].items():
    print()
    print(f"[{label}]  seed              왼(+)   오(-)    차이")
    Ls, Rs = [], []
    for n in names:
        L, R = left_right(n)
        Ls.append(L); Rs.append(R)
        sid = n.split("#")[1]
        print(f"   {sid:<16} {L:6.1f} {R:6.1f} {L - R:+7.1f}")
    Ls, Rs = np.array(Ls), np.array(Rs)
    print(f"   {'평균':<14} {Ls.mean():6.1f} {Rs.mean():6.1f} {Ls.mean() - Rs.mean():+7.1f}")
    print(f"   개별 |왼-오| 평균: {np.abs(Ls - Rs).mean():.1f}%p")

# 결론: 다중 seed 평균에서 좌우 차이 baseline +2.2%p / gSDE +0.1%p → 좌우 편향 사실상 없음.